<a href="https://colab.research.google.com/github/ntpism17/Thai_Election_OCR/blob/main/%E0%B8%A7%E0%B8%B4%E0%B9%80%E0%B8%84%E0%B8%A3%E0%B8%B2%E0%B8%B0%E0%B8%AB%E0%B9%8C%E0%B8%A3%E0%B8%B2%E0%B8%A2%E0%B9%84%E0%B8%94%E0%B9%89%E0%B8%84%E0%B8%A3%E0%B8%B1%E0%B8%A7%E0%B9%80%E0%B8%A3%E0%B8%B7%E0%B8%AD%E0%B8%99%E0%B9%83%E0%B8%99%E0%B8%9B%E0%B8%A3%E0%B8%B0%E0%B9%80%E0%B8%97%E0%B8%A8%E0%B9%84%E0%B8%97%E0%B8%A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EDA — รายได้ครัวเรือนไทย ปี 2566
**ข้อมูล:** สำนักงานสถิติแห่งชาติ | 77 จังหวัด | 10 กลุ่มอาชีพ  
**Visualizations:** Plotly (interactive) — hover เพื่อดูตัวเลข  
---

## Step 1 — Load & Clean Data

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "colab"

FONT_FAMILY = "Garuda, Tahoma, sans-serif"
pio.templates["thai"] = go.layout.Template(
    layout=dict(font=dict(family=FONT_FAMILY))
)
pio.templates.default = "plotly+thai"

df = pd.read_csv("income_per_family.csv", encoding="utf-8-sig")
df = df.drop(columns=["attribute", "source", "unit"])

df["value"] = pd.to_numeric(df["value"], errors="coerce")

df_negative = df[df["value"] < 0].copy()
df_zero     = df[df["value"] == 0].copy()

aggregate_rows = ["ทั่วประเทศ", "ภาคเหนือ", "ภาคตะวันออกเฉียงเหนือ", "ภาคใต้", "ภาคกลาง", "กรุงเทพมหานครและปริมณฑล"]
df_agg = df[df["province"].isin(aggregate_rows)].copy()
df_province = df[~df["province"].isin(aggregate_rows)].copy()

total = df_province[df_province["source_income1"] == "รายได้ทั้งสิ้นต่อเดือน"].copy()

if "ทั่วประเทศ" in df["province"].values:
    national_row = df[(df["province"] == "ทั่วประเทศ") & (df["source_income1"] == "รายได้ทั้งสิ้นต่อเดือน")]
    NATIONAL_MEAN = national_row["value"].mean()
else:
    NATIONAL_MEAN = total["value"].mean()
    print("⚠️ Warning: ไม่พบแถว 'ทั่วประเทศ' คำนวณเป็น Unweighted Mean")

print(f"NATIONAL_MEAN: ฿{NATIONAL_MEAN:,.0f}")

labels_short = {
    "คนงานเกษตรป่าไม้และประมง"                               : "คนงานเกษตร",
    "คนงานด้านการขนส่งและงานพื้นฐาน"                         : "ขนส่ง/งานพื้นฐาน",
    "ผู้ไม่ได้ปฏิบัติงานเชิงเศรษฐกิจ"                        : "ไม่ได้ทำงาน",
    "ประมงป่าไม้ล่าสัตว์หาของป่าบริการทางการเกษตร"            : "ประมง/ป่าไม้",
    "ผู้ปฏิบัติงานในกระบวนการผลิตก่อสร้างและเหมืองแร่"        : "ผลิต/ก่อสร้าง",
    "เสมียนพนักงานขายและให้บริการ"                            : "พนักงานขาย/บริการ",
    "ปลูกพืช/เลี้ยงสัตว์/เพาะเลี้ยง_ส่วนใหญ่เช่าที่ดิน/ทำฟรี": "เกษตร(เช่าที่ดิน)",
    "ปลูกพืช/เลี้ยงสัตว์/เพาะเลี้ยง_ส่วนใหญ่เป็นเจ้าของที่ดิน": "เกษตร(เจ้าของที่ดิน)",
    "ผู้ประกอบธุรกิจของตนเองที่ไม่ใช่การเกษตร"               : "ผู้ประกอบการ",
    "ผู้จัดการนักวิชาการและผู้ปฏิบัติงานวิชาชีพ"             : "Manager/นักวิชาชีพ",
}
total["class_short"] = total["soc_eco_class2"].map(labels_short).fillna(total["soc_eco_class2"])

region_map = {
    "กรุงเทพมหานคร": "กทม.+ปริมณฑล", "นนทบุรี": "กทม.+ปริมณฑล",
    "ปทุมธานี": "กทม.+ปริมณฑล",      "สมุทรปราการ": "กทม.+ปริมณฑล",
    "สมุทรสาคร": "กทม.+ปริมณฑล",     "นครปฐม": "กทม.+ปริมณฑล",
    "ชลบุรี": "ภาคตะวันออก",    "ระยอง": "ภาคตะวันออก",
    "ฉะเชิงเทรา": "ภาคตะวันออก","จันทบุรี": "ภาคตะวันออก",
    "ตราด": "ภาคตะวันออก",      "ปราจีนบุรี": "ภาคตะวันออก",
    "นครนายก": "ภาคตะวันออก",   "สระแก้ว": "ภาคตะวันออก",
    "เชียงใหม่": "ภาคเหนือ", "เชียงราย": "ภาคเหนือ", "ลำปาง": "ภาคเหนือ",
    "ลำพูน": "ภาคเหนือ",     "แม่ฮ่องสอน": "ภาคเหนือ","น่าน": "ภาคเหนือ",
    "พะเยา": "ภาคเหนือ",     "แพร่": "ภาคเหนือ",      "อุตรดิตถ์": "ภาคเหนือ",
    "ตาก": "ภาคเหนือ",       "สุโขทัย": "ภาคเหนือ",   "พิษณุโลก": "ภาคเหนือ",
    "เพชรบูรณ์": "ภาคเหนือ", "กำแพงเพชร": "ภาคเหนือ", "นครสวรรค์": "ภาคเหนือ",
    "พิจิตร": "ภาคเหนือ",    "อุทัยธานี": "ภาคเหนือ",
    "ขอนแก่น": "ภาคอีสาน",  "อุดรธานี": "ภาคอีสาน",  "นครราชสีมา": "ภาคอีสาน",
    "บึงกาฬ": "ภาคอีสาน",   "เลย": "ภาคอีสาน",       "หนองคาย": "ภาคอีสาน",
    "หนองบัวลำภู": "ภาคอีสาน","สกลนคร": "ภาคอีสาน",  "นครพนม": "ภาคอีสาน",
    "กาฬสินธุ์": "ภาคอีสาน", "มหาสารคาม": "ภาคอีสาน","ร้อยเอ็ด": "ภาคอีสาน",
    "ชัยภูมิ": "ภาคอีสาน",   "บุรีรัมย์": "ภาคอีสาน", "สุรินทร์": "ภาคอีสาน",
    "ศรีสะเกษ": "ภาคอีสาน",  "อุบลราชธานี": "ภาคอีสาน","ยโสธร": "ภาคอีสาน",
    "อำนาจเจริญ": "ภาคอีสาน","มุกดาหาร": "ภาคอีสาน",
    "ราชบุรี": "ภาคตะวันตก",    "กาญจนบุรี": "ภาคตะวันตก",
    "สมุทรสงคราม": "ภาคตะวันตก","เพชรบุรี": "ภาคตะวันตก",
    "ประจวบคีรีขันธ์": "ภาคตะวันตก",
    "สุพรรณบุรี": "ภาคกลาง", "พระนครศรีอยุธยา": "ภาคกลาง",
    "อ่างทอง": "ภาคกลาง",    "ลพบุรี": "ภาคกลาง",
    "สิงห์บุรี": "ภาคกลาง",  "ชัยนาท": "ภาคกลาง",    "สระบุรี": "ภาคกลาง",
    "สุราษฎร์ธานี": "ภาคใต้", "นครศรีธรรมราช": "ภาคใต้","กระบี่": "ภาคใต้",
    "พังงา": "ภาคใต้",        "ภูเก็ต": "ภาคใต้",       "ตรัง": "ภาคใต้",
    "สตูล": "ภาคใต้",         "สงขลา": "ภาคใต้",        "ปัตตานี": "ภาคใต้",
    "ยะลา": "ภาคใต้",         "นราธิวาส": "ภาคใต้",     "ชุมพร": "ภาคใต้",
    "ระนอง": "ภาคใต้",        "พัทลุง": "ภาคใต้"
}
total["region"] = total["province"].map(region_map).fillna("Unmatched")

unmatched_provinces = total[total["region"] == "Unmatched"]["province"].unique()
if len(unmatched_provinces) > 0:
    print(f"พบจังหวัดที่ไม่อยู่ใน Mapping: {unmatched_provinces}")

print("\nData ready")

⚠️ Warning: ไม่พบแถว 'ทั่วประเทศ' คำนวณเป็น Unweighted Mean
NATIONAL_MEAN: ฿25,327

Data ready


# Plot 1 — รายได้ตามกลุ่มอาชีพ
### Chart Justification
เลือก **Horizontal Bar Chart** เพราะ label ชื่ออาชีพยาว การวางแนวนอนอ่านได้ชัดกว่าแนวตั้ง
เรียงจากน้อย → มาก เพื่อให้เห็น ranking ทันที
สี แดง / ส้ม / เขียว = ต่ำกว่า / ใกล้เคียง / สูงกว่าค่าเฉลี่ยชาติ ฿25,327

In [ ]:
by_class = total.groupby("class_short")["value"].mean().sort_values().reset_index()
by_class.columns = ["อาชีพ", "รายได้เฉลี่ย"]

def income_color(v):
    if v < 20000:   return "#E8593C"
    elif v < 30000: return "#EF9F27"
    else:           return "#1D9E75"

colors = [income_color(v) for v in by_class["รายได้เฉลี่ย"]]

fig1 = go.Figure()
fig1.add_trace(go.Bar(
    x=by_class["รายได้เฉลี่ย"],
    y=by_class["อาชีพ"],
    orientation="h",
    marker_color=colors,
    text=[f"฿{v:,.0f}" for v in by_class["รายได้เฉลี่ย"]],
    textposition="outside",
    hovertemplate="%{y}<br>รายได้เฉลี่ย: ฿%{x:,.0f}<extra></extra>",
))

fig1.add_vline(
    x=NATIONAL_MEAN,
    line_dash="dash", line_color="#555", line_width=1.5,
    annotation_text=f"ค่าเฉลี่ย 77 จ. ฿{NATIONAL_MEAN:,.0f}",
    annotation_position="bottom right",
    annotation_font_size=11,
)

fig1.update_layout(
    title=dict(text="รายได้เฉลี่ยต่อเดือนของครัวเรือน จำแนกตามกลุ่มอาชีพ (เฉลี่ยจาก 77 จ.) — ปี 2566", font_size=16),
    xaxis_title="รายได้เฉลี่ย (บาท/เดือน)",
    yaxis_title="",
    height=500,
    margin=dict(l=180, r=140, t=60, b=40),
    plot_bgcolor="white",
    xaxis=dict(gridcolor="#eee", range=[0, by_class["รายได้เฉลี่ย"].max() * 1.15]),
)
fig1.show()

### Policy Recommendation
ช่องว่าง Manager (฿48,293) vs คนงานเกษตร (฿14,983) = **3.22×**

- นโยบายค่าแรงขั้นต่ำครอบคลุมเฉพาะกลุ่ม **ลูกจ้าง** ซึ่งเป็นแค่ 5 จาก 10 กลุ่มอาชีพ
- กลุ่มเกษตรกรพึ่งค่าจ้างเพียง 6% → ต้องการ **ประกันราคาสินค้า + market access** ไม่ใช่นโยบายแรงงาน





# PLOT 2: ช่องว่างรายได้ภายในแต่ละประเภทผู้ประกอบการ

###จัด group ตาม soc_eco_class1 → เห็นช่องว่างภายในกลุ่ม ลูกจ้าง/เกษตร/ผู้ประกอบการ

In [ ]:
import plotly.graph_objects as go

by_class1_class2 = (
    total.groupby(["class_short", "soc_eco_class1"])["value"]
    .mean().round(0).reset_index()
)
by_class1_class2.columns = ["อาชีพ", "ประเภทผู้ประกอบการ", "รายได้เฉลี่ย"]

class1_short = {
    "ลูกจ้าง": "ลูกจ้าง",
    "ผู้ถือครองทำการเกษตร/เพาะเลี้ยง": "เกษตรกร",
    "ผู้ประกอบธุรกิจของตนเองที่ไม่ใช่การเกษตร": "ผู้ประกอบการ",
    "ผู้ไม่ได้ปฏิบัติงานเชิงเศรษฐกิจ": "ไม่ได้ทำงาน",
}
by_class1_class2["กลุ่มผู้ประกอบการ"] = (
    by_class1_class2["ประเภทผู้ประกอบการ"].map(class1_short)
    .fillna(by_class1_class2["ประเภทผู้ประกอบการ"])
)
by_class1_class2 = by_class1_class2.sort_values(["กลุ่มผู้ประกอบการ", "รายได้เฉลี่ย"])

occ_colors = {
    "Manager/นักวิชาชีพ": "#534AB7", "พนักงานขาย/บริการ":  "#3B8BD4",
    "ผลิต/ก่อสร้าง":     "#1D9E75", "คนงานเกษตร":        "#EF9F27",
    "ขนส่ง/งานพื้นฐาน":  "#E8593C", "เกษตร(เจ้าของที่ดิน)": "#D4537E",
    "เกษตร(เช่าที่ดิน)":   "#F97316", "ประมง/ป่าไม้":       "#888780",
    "ผู้ประกอบการ":       "#10B981", "ไม่ได้ทำงาน":        "#A78BFA",
}
class1_order = ["ไม่ได้ทำงาน", "ผู้ประกอบการ", "เกษตรกร", "ลูกจ้าง"]
fig_dumbbell = go.Figure()

for grp in class1_order:
    df_grp = by_class1_class2[by_class1_class2["กลุ่มผู้ประกอบการ"] == grp]
    if len(df_grp) > 1: # ถ้ามีมากกว่า 1 อาชีพให้ขีดเส้นเชื่อม
        fig_dumbbell.add_trace(go.Scatter(
            x=[df_grp["รายได้เฉลี่ย"].min(), df_grp["รายได้เฉลี่ย"].max()],
            y=[grp, grp],
            mode="lines",
            line=dict(color="#E2E8F0", width=6),
            showlegend=False,
            hoverinfo="skip"
        ))

for occ in by_class1_class2["อาชีพ"].unique():
    df_occ = by_class1_class2[by_class1_class2["อาชีพ"] == occ]
    fig_dumbbell.add_trace(go.Scatter(
        x=df_occ["รายได้เฉลี่ย"],
        y=df_occ["กลุ่มผู้ประกอบการ"],
        mode="markers+text",
        marker=dict(size=14, color=occ_colors.get(occ, "#333"), line=dict(color="white", width=1)),
        name=occ,
        text=df_occ["รายได้เฉลี่ย"].apply(lambda x: f"฿{x/1000:,.0f}k"), # ย่อตัวเลขเป็น k ให้ดูคลีน
        textposition="top center",
        textfont=dict(size=10, color="#555"),
        hovertemplate=f"<b>{occ}</b><br>รายได้: ฿%{{x:,.0f}}<extra></extra>"
    ))

fig_dumbbell.add_vline(
    x=NATIONAL_MEAN, line_dash="dash", line_color="#555", line_width=1.5,
    annotation_text=f"ค่าเฉลี่ย ฿{NATIONAL_MEAN:,.0f}", annotation_position="top right"
)

fig_dumbbell.update_layout(
    title=(
        "ช่องว่างรายได้ภายในแต่ละสถานะการทำงาน — ปี 2566<br>"
        "<sup>กลุ่มลูกจ้างมีช่องว่างกว้างที่สุด: ความห่างระหว่างเส้นแสดงถึงความเหลื่อมล้ำภายในกลุ่ม</sup>"
    ),
    height=450,
    plot_bgcolor="white",
    xaxis=dict(gridcolor="#F1F5F9", title="รายได้เฉลี่ย (฿/เดือน)", range=[0, 55000]),
    yaxis=dict(title=""),
    margin=dict(l=120, r=40, t=80, b=40),
    legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5, title="")
)
fig_dumbbell.show()

##Policy Recommendation



* กลุ่มลูกจ้าง (เร่งด่วนที่สุด): ช่องว่าง 3.2 เท่าภายในกลุ่มเดียวกันบอกว่าค่าแรงขั้นต่ำอย่างเดียวไม่พอ ต้องเสริมด้วย vocational upskilling โดยเฉพาะกลุ่มขนส่ง/งานพื้นฐานและคนงานเกษตรที่เป็นลูกจ้าง เพื่อให้ขยับขึ้นไปสู่อาชีพที่รายได้สูงกว่าได้ ควรเน้น digital skills และ technical certification ที่ตลาดต้องการ

* กลุ่มเกษตรกร (ยกระดับทั้งระบบ): ปัญหาไม่ใช่ gap ภายในกลุ่ม แต่คือรายได้ต่ำทั้งระบบ นโยบายค่าแรงขั้นต่ำไม่ช่วยเพราะเกษตรกรไม่ใช่ลูกจ้าง ต้องใช้ประกันราคาสินค้าเกษตร, เข้าถึง market access / e-commerce, และลดต้นทุน logistics จากไร่นาสู่ตลาด

* กลุ่มผู้ประกอบการ (เสริมให้แข็งแกร่ง): รายได้ดีแล้ว แต่ควรสนับสนุน SME financing + ลดภาระ compliance เพื่อให้ธุรกิจขนาดเล็กอยู่รอดและเติบโต ไม่ใช่แค่ธุรกิจขนาดกลาง-ใหญ่

* กลุ่มไม่ได้ทำงาน (สร้างโอกาส ไม่ใช่แค่แจกเงิน): การเพิ่มเงินสวัสดิการเป็น short-term fix เท่านั้น ต้องสร้างโอกาสการจ้างงานและ re-entry program สำหรับผู้สูงอายุ/ผู้พิการ/แม่บ้านที่พร้อมกลับเข้าตลาดแรงงาน



#plot 3 - Map
Chart Justification
เลือกนำเสนอแผนที่สองรูปนี้คู่กัน เพราะมันคือการเล่าเรื่องแบบ Macro to Micro (จากภาพใหญ่สู่ภาพย่อย) แผนที่ A (รายได้เฉลี่ย) ทำให้เราเห็นว่า 'เค้กของจังหวัดไหนก้อนใหญ่กว่ากัน' แต่แผนที่ B (ความเหลื่อมล้ำ) จะเป็นตัวบอกความจริงว่า 'เค้กก้อนนั้นถูกแบ่งให้คนในจังหวัดอย่างเป็นธรรมหรือไม่' การใช้กราฟสองตัวนี้ร่วมกัน จึงให้มุมมองที่ครบถ้วนทั้งมิติการเติบโตทางเศรษฐกิจ (Economic Growth) และมิติความยุติธรรมทางสังคม (Social Equity)

In [ ]:
province_coords = {
    "กรุงเทพมหานคร": (13.7563, 100.5018),
    "กระบี่": (8.0863, 98.9063),
    "กาญจนบุรี": (14.0043, 99.5328),
    "กาฬสินธุ์": (16.4315, 103.5060),
    "กำแพงเพชร": (16.4828, 99.5227),
    "ขอนแก่น": (16.4322, 102.8236),
    "จันทบุรี": (12.6113, 102.1040),
    "ฉะเชิงเทรา": (13.6904, 101.0780),
    "ชลบุรี": (13.3611, 100.9847),
    "ชัยนาท": (15.1851, 100.1251),
    "ชัยภูมิ": (15.8068, 102.0316),
    "ชุมพร": (10.4930, 99.1800),
    "เชียงราย": (19.9105, 99.8406),
    "เชียงใหม่": (18.7883, 98.9853),
    "ตรัง": (7.5564, 99.6114),
    "ตราด": (12.2428, 102.5175),
    "ตาก": (16.8840, 99.1259),
    "นครนายก": (14.2069, 101.2133),
    "นครปฐม": (13.8196, 100.0644),
    "นครพนม": (17.3920, 104.7695),
    "นครราชสีมา": (14.9799, 102.0978),
    "นครศรีธรรมราช": (8.4304, 99.9631),
    "นครสวรรค์": (15.6930, 100.1225),
    "นนทบุรี": (13.8622, 100.5144),
    "นราธิวาส": (6.4255, 101.8253),
    "น่าน": (18.7756, 100.7730),
    "บึงกาฬ": (18.3609, 103.6466),
    "บุรีรัมย์": (14.9931, 103.1029),
    "ปทุมธานี": (14.0208, 100.5250),
    "ประจวบคีรีขันธ์": (11.8126, 99.7975),
    "ปราจีนบุรี": (14.0579, 101.3713),
    "ปัตตานี": (6.8699, 101.2508),
    "พระนครศรีอยุธยา": (14.3692, 100.5877),
    "พะเยา": (19.1664, 99.9019),
    "พังงา": (8.4509, 98.5225),
    "พิจิตร": (16.4429, 100.3487),
    "พิษณุโลก": (16.8211, 100.2659),
    "เพชรบุรี": (13.1112, 99.9391),
    "เพชรบูรณ์": (16.4190, 101.1591),
    "แพร่": (18.1445, 100.1403),
    "ภูเก็ต": (7.8804, 98.3923),
    "มหาสารคาม": (16.1851, 103.3008),
    "มุกดาหาร": (16.5424, 104.7235),
    "แม่ฮ่องสอน": (19.3020, 97.9654),
    "ยโสธร": (15.7944, 104.1449),
    "ยะลา": (6.5414, 101.2803),
    "ร้อยเอ็ด": (16.0538, 103.6520),
    "ระนอง": (9.9528, 98.6085),
    "ระยอง": (12.6814, 101.2816),
    "ราชบุรี": (13.5283, 99.8134),
    "ลพบุรี": (14.7995, 100.6534),
    "ลำปาง": (18.2888, 99.4910),
    "ลำพูน": (18.5744, 98.9862),
    "เลย": (17.4860, 101.7223),
    "ศรีสะเกษ": (15.1186, 104.3220),
    "สกลนคร": (17.1545, 104.1348),
    "สงขลา": (7.1896, 100.5945),
    "สตูล": (6.6238, 100.0673),
    "สมุทรปราการ": (13.5991, 100.5998),
    "สมุทรสงคราม": (13.4098, 100.0024),
    "สมุทรสาคร": (13.5475, 100.2744),
    "สระแก้ว": (13.8240, 102.0645),
    "สระบุรี": (14.5289, 100.9103),
    "สิงห์บุรี": (14.8936, 100.3967),
    "สุโขทัย": (17.0100, 99.8265),
    "สุพรรณบุรี": (14.4744, 100.1177),
    "สุราษฎร์ธานี": (9.1382, 99.3217),
    "สุรินทร์": (14.8828, 103.4937),
    "หนองคาย": (17.8783, 102.7420),
    "หนองบัวลำภู": (17.2042, 102.4260),
    "อ่างทอง": (14.5896, 100.4549),
    "อำนาจเจริญ": (15.8656, 104.6258),
    "อุดรธานี": (17.4156, 102.7872),
    "อุตรดิตถ์": (17.6200, 100.0993),
    "อุทัยธานี": (15.3835, 100.0245),
    "อุบลราชธานี": (15.2286, 104.8564),
    "พัทลุง": (7.6167, 100.0833)
}
prov_avg = (total.groupby(["province","region"])["value"]
            .mean().round(0).reset_index()
            .sort_values("value", ascending=False)
            .reset_index(drop=True))
prov_avg["lat"] = prov_avg["province"].map(lambda p: province_coords.get(p, (None,None))[0])
prov_avg["lon"] = prov_avg["province"].map(lambda p: province_coords.get(p, (None,None))[1])
prov_avg = prov_avg.dropna(subset=["lat","lon"])

region_colors = {
    "กทม.+ปริมณฑล": "#534AB7", "ภาคกลาง": "#D4537E",
    "ภาคตะวันตก":   "#888780",  "ภาคตะวันออก": "#1D9E75",
    "ภาคอีสาน":     "#E8593C",  "ภาคใต้": "#EF9F27",
    "ภาคเหนือ":     "#3B8BD4",
}

fig_map1 = px.scatter_geo(
    prov_avg,
    lat="lat", lon="lon",
    color="region",
    size="value",
    size_max=28,
    hover_name="province",
    hover_data={"value": ":,.0f", "region": True, "lat": False, "lon": False},
    color_discrete_map=region_colors,
    labels={"value": "รายได้เฉลี่ย (฿/เดือน)", "region": "ภาค"},
    title="แผนที่รายได้เฉลี่ยต่อเดือนของครัวเรือน รายจังหวัด — ปี 2566<br>"
          "<sup>ขนาดวงกลม = รายได้เฉลี่ย | สี = ภาค | hover เพื่อดูตัวเลข</sup>",
)

fig_map1.update_geos(
    scope="asia",
    center=dict(lat=13.2, lon=101.0),
    projection_scale=12,
    showland=True,     landcolor="#f5f5f0",
    showocean=True,    oceancolor="#e8f0f8",
    showcountries=True, countrycolor="#ccc",
    showlakes=False,
    showcoastlines=True, coastlinecolor="#ccc",
    lataxis=dict(range=[5.0, 21.0]),
    lonaxis=dict(range=[96.5, 106.0]),
)
fig_map1.update_layout(
    height=750,
    margin=dict(l=0, r=0, t=80, b=0),
    legend=dict(
        title="ภาค", orientation="h",
        yanchor="bottom", y=-0.05,
        xanchor="center", x=0.5,
    ),
)
fig_map1.show()

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px


MANAGER = "ผู้จัดการนักวิชาการและผู้ปฏิบัติงานวิชาชีพ"
AGRI    = "คนงานเกษตรป่าไม้และประมง"
pivot = total.pivot_table(index="province", columns="soc_eco_class2", values="value")
ineq = pd.DataFrame({
    "avg_income" : pivot.mean(axis=1),
    "cv"         : (pivot.std(axis=1) / pivot.mean(axis=1) * 100),
    "range_income": pivot.max(axis=1) - pivot.replace(0, np.nan).min(axis=1),
    "manager"    : pivot[MANAGER],
    "agri"       : pivot.replace(0, np.nan)[AGRI],
}).dropna(subset=["cv"])


ineq["ratio"]  = (ineq["manager"] / ineq["agri"]).round(2)
ineq["จังหวัด"] = ineq.index
ineq["lat"] = ineq["จังหวัด"].map(lambda p: province_coords.get(p, (None,None))[0])
ineq["lon"] = ineq["จังหวัด"].map(lambda p: province_coords.get(p, (None,None))[1])
ineq_map = ineq.dropna(subset=["lat","lon"]).copy()

fig_map2 = px.scatter_geo(
    ineq_map,
    lat="lat", lon="lon",
    color="cv",
    size="range_income",
    size_max=26,
    hover_name="จังหวัด",
    hover_data={
        "cv": ":.1f",
        "avg_income": ":, .0f",
        "ratio": ":.2f",
        "range_income": ":, .0f",
        "lat": False, "lon": False,
    },
    color_continuous_scale="YlOrRd",
    labels={
        "cv": "CV (%)",
        "avg_income": "รายได้เฉลี่ย (฿)",
        "ratio": "อัตราส่วน Manager/เกษตร",
        "range_income": "ช่วงห่าง max-min (฿)"
    },
    title="แผนที่ความเหลื่อมล้ำภายในจังหวัด (CV) — ปี 2566<br>"
          "<sup>สีเข้ม = CV สูง (เหลื่อมล้ำมาก) | ขนาดวงกลม = ช่วงห่างรายได้ max–min</sup>",
)

fig_map2.update_geos(
    scope="asia",
    center=dict(lat=13.2, lon=101.0),
    projection_scale=12,
    showland=True, landcolor="#f5f5f0",
    showocean=True, oceancolor="#e8f0f8",
    showcountries=True, countrycolor="#ccc",
    showcoastlines=True, coastlinecolor="#ccc",
    lataxis=dict(range=[5.0, 21.0]),
    lonaxis=dict(range=[96.5, 106.0]),
)
fig_map2.update_layout(height=750, margin=dict(l=0, r=0, t=80, b=0))
fig_map2.show()

##Policy Recommendation
1. นโยบายมิติพื้นที่: กระจายความเจริญสู่ภูมิภาค (อิงจาก Map A)
จาก Map A เราจะเห็นชัดเจนว่าเม็ดเงินและรายได้ไปกระจุกตัวอยู่ที่ กทม. ปริมณฑล ภาคตะวันออก (พื้นที่อุตสาหกรรม EEC) และจังหวัดท่องเที่ยวหลัก (เช่น ภูเก็ต) ในขณะที่ภาคอีสานและเหนือยังมีรายได้เฉลี่ยที่ต่ำกว่ามาก

  * สร้าง Regional Economic Hubs: รัฐควรมีมาตรการจูงใจทางภาษีหรือเงินอุดหนุน เพื่อดึงดูดให้บริษัทเอกชนขยายฐานการผลิตหรือตั้งสำนักงานสาขาในจังหวัดหัวเมืองรองในภาคเหนือและอีสาน เพื่อลดการกระจุกตัวของงานทักษะสูงในเมืองหลวง
  * ยกระดับโครงสร้างพื้นฐานข้ามภูมิภาค: ลงทุนในระบบขนส่ง (Rail & Road) และอินเทอร์เน็ตความเร็วสูง เพื่อให้จังหวัดที่อยู่ห่างไกลสามารถเชื่อมต่อกับ Supply Chain หลักของประเทศได้ง่ายขึ้น ลดต้นทุนโลจิสติกส์ให้ผู้ประกอบการท้องถิ่น

2. นโยบายมิติโครงสร้างอาชีพ: ยกระดับฐานราก (อิงจาก Map B)
Map B เผยให้เห็น "ช่องว่าง" (Gap) ที่กว้างมากระหว่างกลุ่มผู้จัดการ/นักวิชาชีพ กับ คนงานเกษตร/แรงงานพื้นฐาน แม้แต่ในจังหวัดที่รายได้เฉลี่ยสูงก็ตาม



  * Upskilling & Reskilling (การยกระดับทักษะ): นโยบายให้ทุนสนับสนุนแรงงานพื้นฐานในการฝึกอบรมทักษะใหม่ๆ ที่ตลาดต้องการ (เช่น ทักษะดิจิทัล หรือช่างเทคนิคขั้นสูง) เพื่อให้พวกเขาสามารถขยับสถานะทางอาชีพและหลุดพ้นจากกับดักค่าแรงขั้นต่ำได้

  * Agri-Tech & Smart Farming: เนื่องจากรายได้ของ "คนงานเกษตร" ต่ำรั้งท้าย นโยบายควรเน้นการให้ความรู้และอุดหนุนเทคโนโลยีการเกษตรแม่นยำ (Precision Agriculture) เพื่อเพิ่มผลผลิตต่อไร่ (Yield) และการแปรรูปสินค้าเกษตรเพื่อสร้างมูลค่าเพิ่ม แทนการขายวัตถุดิบราคาถูก

  * มาตรการทางภาษีและสวัสดิการ (Redistribution): จัดเก็บภาษีในอัตราก้าวหน้าอย่างมีประสิทธิภาพมากขึ้น และนำรายได้มาจัดสรรเป็น "สวัสดิการพุ่งเป้า" (Targeted Welfare) เช่น เงินอุดหนุนเด็กเล็ก หรือการรักษาพยาบาลฟรี เพื่อลดภาระค่าใช้จ่ายของครัวเรือนรายได้น้อย



# Plot 4 — โครงสร้างรายได้แต่ละกลุ่มอาชีพ
### Chart Justification
เลือก **100% Stacked Bar** เพราะต้องการเปรียบเทียบ **สัดส่วน** ไม่ใช่ตัวเลขสัมบูรณ์
เรียงจากพึ่งค่าจ้างน้อย → มาก ทำให้เห็น gradient ของความเปราะบางตั้งแต่กลุ่มเกษตรไปถึงลูกจ้าง
กลุ่มที่อยู่บนสุด = เปราะบางที่สุด เพราะไม่มีรายได้ประจำเป็นฐาน



In [ ]:
income_sources = {
    "ค่าจ้างและเงินเดือน"           : "ค่าจ้าง/เงินเดือน",
    "กำไรสุทธิจากการทำธุรกิจ"        : "กำไรธุรกิจ",
    "กำไรสุทธิจากการทำการเกษตร"      : "กำไรเกษตร",
    "รายได้ที่ไม่เป็นตัวเงิน"         : "รายได้ Non-cash",
    "เงินที่ได้รับเป็นการช่วยเหลือ"   : "เงินช่วยเหลือ/สวัสดิการ",
    "รายได้จากทรัพย์สิน"              : "รายได้ทรัพย์สิน",
    "รายได้ไม่ประจำ (ที่เป็นตัวเงิน)": "รายได้ไม่ประจำ",
}
source_colors = {
    "ค่าจ้าง/เงินเดือน"        : "#534AB7",
    "กำไรธุรกิจ"               : "#1D9E75",
    "กำไรเกษตร"                : "#EF9F27",
    "รายได้ Non-cash"           : "#3B8BD4",
    "เงินช่วยเหลือ/สวัสดิการ"  : "#E8593C",
    "รายได้ทรัพย์สิน"           : "#D4537E",
    "รายได้ไม่ประจำ"            : "#888780",
}
struct = (df[df["source_income3"].isin(income_sources.keys())]
          .copy())
struct["source_short"] = struct["source_income3"].map(income_sources)
struct["class_short"]  = struct["soc_eco_class2"].map(labels_short)
struct_grp = (struct.groupby(["class_short","source_short"])["value"]
              .mean().round(0).reset_index())
total_per_class = struct_grp.groupby("class_short")["value"].sum()
struct_grp["total"]   = struct_grp["class_short"].map(total_per_class)
struct_grp["pct"]     = (struct_grp["value"] / struct_grp["total"] * 100).round(1)
wage_pct = (struct_grp[struct_grp["source_short"] == "ค่าจ้าง/เงินเดือน"]
            .set_index("class_short")["pct"])
class_order = wage_pct.sort_values(ascending=True).index.tolist()
fig2 = px.bar(
    struct_grp,
    x="pct", y="class_short",
    color="source_short",
    orientation="h",
    barmode="stack",
    color_discrete_map=source_colors,
    category_orders={"class_short": class_order},
    text="pct",
    labels={"pct": "สัดส่วน (%)", "class_short": "",
            "source_short": "แหล่งรายได้"},
    title="โครงสร้างแหล่งที่มาของรายได้ จำแนกตามกลุ่มอาชีพ — ปี 2566",
)
fig2.update_traces(
    texttemplate="%{text:.0f}%",
    textposition="inside",
    insidetextanchor="middle",
    selector=lambda t: t.x.max() > 5,
)
fig2.add_vline(x=50, line_dash="dot", line_color="#aaa",
                 line_width=1, annotation_text="50%",
                 annotation_position="top")
fig2.update_layout(
    height=500, plot_bgcolor="white",
    xaxis=dict(range=[0,100], gridcolor="#eee", ticksuffix="%"),
    margin=dict(l=190, r=60, t=90, b=40),
    legend_title_text="ประเภทรายได้",
)
fig2.show()

### Policy Recommendation
แต่ละกลุ่มอาชีพต้องการนโยบายคนละชุด ไม่ใช่มาตรการเดียวใช้ได้ทุกกลุ่ม

| กลุ่มอาชีพ | แหล่งรายได้หลัก | นโยบายที่ตรงจุด |
|-----------|----------------|----------------|
| เกษตรกร | กำไรเกษตร 47–55% ค่าจ้างแค่ 6% | ประกันราคาสินค้า + market access |
| ลูกจ้าง | ค่าจ้าง 67–78% | ค่าแรงขั้นต่ำ + upskilling |
| เจ้าของกิจการ | กำไรธุรกิจ 73% | SME financing + ลดต้นทุน |
| ไม่ได้ทำงาน | สวัสดิการ 58% | สร้างโอกาสการจ้างงาน ไม่ใช่เพิ่มเงินช่วยเหลือ |



# plot 5 — Bubble chart: ระบุจังหวัดที่ inequality สูง
### Chart Justification
เลือก **Bubble Chart (4 Quadrant)** เพราะต้องแสดง 3 มิติพร้อมกัน
ได้แก่ รายได้เฉลี่ย (X), ความเหลื่อมล้ำภายใน CV (Y) และขนาดช่วงรายได้ (ขนาดฟอง)
การแบ่ง 4 quadrant ช่วยจัดกลุ่มจังหวัดตาม policy priority ได้ทันทีโดยไม่ต้องอ่านทุกจุด


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

MANAGER = "ผู้จัดการนักวิชาการและผู้ปฏิบัติงานวิชาชีพ"
AGRI    = "คนงานเกษตรป่าไม้และประมง"

pivot = total.pivot_table(index="province", columns="soc_eco_class2", values="value")


ineq = pd.DataFrame({
    "avg_income"  : pivot.mean(axis=1),
    "cv"          : (pivot.std(axis=1) / pivot.mean(axis=1) * 100),
    "range_income": pivot.max(axis=1) - pivot.replace(0, np.nan).min(axis=1),
    "manager"     : pivot[MANAGER],
    "agri"        : pivot.replace(0, np.nan)[AGRI],
}).dropna(subset=["cv"])

ineq["ratio"]  = (ineq["manager"] / ineq["agri"]).round(2)
ineq["region"] = ineq.index.map(region_map).fillna("ภาคกลาง")
ineq = ineq.reset_index().rename(columns={"province": "จังหวัด"})


median_income = ineq["avg_income"].median()
median_cv     = ineq["cv"].median()


def quadrant(row):
    hi_income = row["avg_income"] >= median_income
    hi_cv     = row["cv"]         >= median_cv

    if hi_income and hi_cv:       return "Q1: รวยแต่เหลื่อมล้ำสูง"
    elif hi_income and not hi_cv: return "Q4: รวยและกระจายรายได้ดี"
    elif not hi_income and hi_cv: return "Q2: จนและเหลื่อมล้ำสูง"
    else:                         return "Q3: จนและกระจายรายได้ดี"

ineq["quadrant"] = ineq.apply(quadrant, axis=1)


quad_colors = {
    "Q1: รวยแต่เหลื่อมล้ำสูง": "#E8593C",
    "Q2: จนและเหลื่อมล้ำสูง": "#EF9F27",
    "Q3: จนและกระจายรายได้ดี": "#888780",
    "Q4: รวยและกระจายรายได้ดี": "#1D9E75"
}

fig_quad = px.scatter(
    ineq,
    x="avg_income",
    y="cv",
    color="quadrant",
    size="range_income",
    size_max=35,
    hover_name="จังหวัด",
    hover_data={
        "quadrant": False,
        "avg_income": ":,.0f",
        "cv": ":.1f",
        "range_income": ":,.0f",
        "region": True
    },
    color_discrete_map=quad_colors,
    labels={
        "avg_income": "รายได้เฉลี่ย (บาท/เดือน)",
        "cv": "ความเหลื่อมล้ำ CV (%)",
        "range_income": "ช่วงห่างรายได้ max-min",
        "region": "ภูมิภาค"
    },
    title="การวิเคราะห์จัดกลุ่มจังหวัด 4 มิติ (Income vs Inequality) — ปี 2566<br>"
          "<sup>เส้นประ = ค่ามัธยฐาน 77 จังหวัด | ขนาดวงกลม = ความกว้างของช่องว่างรายได้</sup>"
)

fig_quad.add_vline(x=median_income, line_dash="dash", line_color="#333", line_width=1.5,
                   annotation_text="Median Income", annotation_position="top right")
fig_quad.add_hline(y=median_cv, line_dash="dash", line_color="#333", line_width=1.5,
                   annotation_text="Median CV", annotation_position="top right")


fig_quad.update_layout(
    height=650,
    plot_bgcolor="white",
    xaxis=dict(gridcolor="#f1f5f9", zeroline=False),
    yaxis=dict(gridcolor="#f1f5f9", zeroline=False),
    margin=dict(l=60, r=40, t=80, b=60),
    legend=dict(
        title="กลุ่มนโยบาย (Policy Priority)",
        orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5
    )
)

fig_quad.show()

### Policy Recommendation
จังหวัดใน 4 quadrant ต้องการนโยบายคนละชุด

| Quadrant | ตัวอย่างจังหวัด | นโยบายที่เหมาะสม |
|----------|----------------|-----------------|
| จน + เหลื่อมล้ำสูง  | นราธิวาส ยะลา แม่ฮ่องสอน | Targeted skill development ลดช่องว่างก่อน |
| จน + เหลื่อมล้ำต่ำ | กำแพงเพชร สงขลา | ยกระดับรายได้โดยรวม เช่น logistics + infrastructure |
| รวย + เหลื่อมล้ำสูง | สมุทรสาคร ราชบุรี | กระจายผลประโยชน์จากการเติบโตให้ทั่วถึงกว่านี้ |
| รวย + เหลื่อมล้ำต่ำ | ภูเก็ต สุราษฎร์ธานี | รักษาโมเดลที่ดีอยู่แล้ว + ขยาย sector ใหม่ |

# Plot 4 — Top 15 / Bottom 15 จังหวัด
### Chart Justification
เลือก **Side-by-side Horizontal Bar (subplot 2 คอลัมน์)** เพราะต้องการเปรียบเทียบขั้วตรงข้าม
(Top 15 vs Bottom 15) พร้อมกันในภาพเดียว การวาง Left = รวย / Right = จน
ทำให้เห็น contrast ของช่องว่างระหว่างจังหวัดได้ทันที
Color gradient เข้ม = สูง / จาง = ต่ำ ช่วย encode ข้อมูลซ้ำอีกชั้นโดยไม่ต้องอ่านตัวเลข

In [ ]:
prov_avg = (total.groupby(["province","region"])["value"]
            .mean().round(0).reset_index()
            .sort_values("value", ascending=False)
            .reset_index(drop=True))
prov_avg["rank"] = prov_avg.index + 1
top15 = prov_avg.head(15).sort_values("value")
bot15 = prov_avg.tail(15).sort_values("value")
fig4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("15 จังหวัดรายได้สูงสุด", "15 จังหวัดรายได้ต่ำสุด"), # แก้ A1, A2
    horizontal_spacing=0.12,
)
for df_sub, col, cmap in [(top15, 1, "Greens"), (bot15, 2, "Reds")]:
    fig4.add_trace(go.Bar(
        x=df_sub["value"],
        y=df_sub["province"],
        orientation="h",
        marker=dict(color=df_sub["value"],
                    colorscale=cmap,
                    showscale=False),
        text=[f"฿{v:,.0f}" for v in df_sub["value"]],
        textposition="outside",
        hovertemplate="%{y}<br>฿%{x:,.0f}<extra></extra>",
    ), row=1, col=col)
for col in [1, 2]:
    fig4.add_vline(x=NATIONAL_MEAN, line_dash="dash",
                   line_color="#777", line_width=1, row=1, col=col)
fig4.update_layout(
    title=dict(
        text="เปรียบเทียบ 15 จังหวัดที่มีรายได้เฉลี่ยสูงสุด และ ต่ำสุด — ปี 2566", # แก้ A3
        font_size=15,
    ),
    height=580,
    showlegend=False,
    plot_bgcolor="white",
)
fig4.update_xaxes(gridcolor="#eee")
fig4.show()

### Policy Recommendation
ช่องว่างระหว่างอันดับ 1 (จันทบุรี ฿50,893) vs อันดับ 77 (นราธิวาส ฿13,965) = **3.64×**

- **Top 15** ส่วนใหญ่อยู่ในโซน EEC + ปริมณฑล + เกษตรพาณิชย์
  → นโยบายควรรักษา momentum และ **กระจายผลประโยชน์** ไปยังแรงงานระดับล่างในพื้นที่
- **Bottom 15** กระจุกที่ภาคเหนือ + ชายแดนใต้ ซึ่งมีบริบทต่างกัน
  → ไม่ควรใช้นโยบายเดียวกัน เช่น ภาคเหนือต้องการ **agricultural value chain**
  ส่วนชายแดนใต้ต้องการ **ความมั่นคงและการลงทุนโครงสร้างพื้นฐาน** ก่อน
- จันทบุรีอันดับ 1 มาจาก **outlier เกษตร(เช่าที่ดิน) ฿181,322**
  → ควรใช้ **median แทน mean** ถ้าต้องการสะท้อนรายได้ของคนส่วนใหญ่ในจังหวัด

# Plot 6 — Heatmap จังหวัด × อาชีพ
### Chart Justification
เลือก **Heatmap** เพราะต้องเปรียบเทียบ 77 × 10 = 770 ค่าพร้อมกันในภาพเดียว
Bar chart จะต้องใช้ 10 กราฟแยกหรือกราฟเดียวที่รกมาก
ความเข้มสีแทนตัวเลข ทำให้ตาจับ pattern ระดับภาค และ outlier ได้เร็วกว่าการอ่านตาราง
เรียงจังหวัดจากรายได้เฉลี่ยต่ำ → สูง ทำให้เห็น gradient ของความเหลื่อมล้ำระหว่างจังหวัดในแนวตั้ง


In [ ]:
pivot_heat = total.pivot_table(index="province", columns="class_short", values="value")
pivot_heat.columns.name = None
pivot_heat.index.name   = None
prov_order = pivot_heat.mean(axis=1).sort_values(ascending=True).index
pivot_heat = pivot_heat.loc[prov_order]
fig6 = go.Figure(go.Heatmap(
    z=pivot_heat.values,
    x=pivot_heat.columns.tolist(),
    y=pivot_heat.index.tolist(),
    colorscale="YlOrRd",
    hovertemplate="%{y} — %{x}<br>฿%{z:,.0f}<extra></extra>",
    colorbar=dict(title="บาท/เดือน", thickness=15),
))
fig6.update_layout(
    title=dict(text="Heatmap แสดงรายได้เฉลี่ย จำแนกตามจังหวัดและกลุ่มอาชีพ — ปี 2566", # แก้ A2
               font_size=15),
    xaxis=dict(tickangle=-35, tickfont_size=11),
    yaxis=dict(tickfont_size=10),
    height=1900,
    margin=dict(l=140, r=100, t=60, b=120),
)
fig6.show()

### Policy Recommendation
- **แถวที่สีเข็มทั้งแถว** (จันทบุรี ปทุมธานี ภูเก็ต) = จังหวัดที่ทุกกลุ่มอาชีพรายได้ดี
  → เศรษฐกิจแข็งแกร่งจริง ไม่ได้ดีเฉพาะกลุ่มใดกลุ่มหนึ่ง
- **แถวที่สีอ่อนทั้งแถว** (นราธิวาส เชียงราย ขอนแก่น) = จังหวัดที่ทุกกลุ่มอาชีพรายได้ต่ำ
  → ต้องการนโยบายยกระดับโครงสร้างเศรษฐกิจทั้งจังหวัด ไม่ใช่แค่กลุ่มใดกลุ่มหนึ่ง
- **แถวที่มีสีต่างกันในแต่ละคอลัมน์** = จังหวัดที่มี dual economy เช่น เลย ยโสธร
  → Manager เข็มมาก แต่คนงานเกษตรสีอ่อน = ช่องว่างสูง ต้องการนโยบาย
  targeted ไม่ใช่ blanket policy